Se requiere obtener información de las `películas` del año 2000 en adelante ordenado de manera descendente por la fecha de lanzamiento, se debe considerar su respectivo `género` y el `idioma` que se encuentra dicha película

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

## 1. Obtenemos las películas y campos que nos interesan (`título`, `tiempo de duración`, `fecha de lanzamiento` y `voto promedio`)

In [0]:
movies_df = spark.read.table("movie_silver.movies").filter(f"file_date = '{v_file_date}'")
movies_filtered_df = (movies_df
    .filter(movies_df.year_release_date >= 2000)
    .select("movie_id", "title", "duration_time", "release_date", "vote_average")
)
display(movies_filtered_df)


movie_id,title,duration_time,release_date,vote_average
12,Finding Nemo,100,2003-05-30,7.6
16,Dancer in the Dark,140,2000-05-17,7.6
20,My Life Without Me,106,2003-03-07,7.2
22,Pirates of the Caribbean: The Curse of the Black Pearl,143,2003-07-09,7.5
24,Kill Bill: Vol. 1,111,2003-10-10,7.7
25,Jarhead,125,2005-11-04,6.6
35,The Simpsons Movie,87,2007-07-25,6.9
38,Eternal Sunshine of the Spotless Mind,108,2004-03-19,7.9
55,Amores perros,154,2000-06-16,7.6
58,Pirates of the Caribbean: Dead Man's Chest,151,2006-06-20,7.0


## 2. Obtener el `nombre del idioma` de la película

In [0]:
movies_languages_df = spark.read.table("movie_silver.movies_languages").filter(f"file_date = '{v_file_date}'")
languages_df = spark.read.table("movie_silver.languages")

movies_languajes_name_df = (
    movies_languages_df.join(languages_df, on="language_id", how="inner")
    .select("movie_id", "language_name", movies_languages_df.language_id)
)

## 3. Añadir `nombre del idioma` al DataFrame

In [0]:
movies_final_df = (
    movies_filtered_df.join(movies_languajes_name_df, on="movie_id", how="inner")
    .orderBy("release_date", asc=False)
    .select("movie_id", "title", "duration_time", "release_date", "vote_average", "language_name", "language_id")
)

display(movies_final_df)

movie_id,title,duration_time,release_date,vote_average,language_name,language_id
10384,Supernova,91,2000-01-14,4.9,English,24574
17908,My Dog Skip,95,2000-01-14,6.5,English,24574
10472,Down to You,91,2000-01-21,4.9,Français,24579
10472,Down to You,91,2000-01-21,4.9,English,24574
22597,The Broken Hearts Club: A Romantic Comedy,94,2000-02-01,6.3,Français,24579
22597,The Broken Hearts Club: A Romantic Comedy,94,2000-02-01,6.3,English,24574
1698,Anatomie,103,2000-02-03,6.1,Deutsch,24576
4234,Scream 3,116,2000-02-03,5.7,English,24574
1698,Anatomie,103,2000-02-03,6.1,Latin,24582
29076,Gun Shy,101,2000-02-04,5.4,English,24574


## 4. Obtener el `género` de la película

In [0]:
movies_genres_df = spark.read.table("movie_silver.movies_genres").filter(f"file_date = '{v_file_date}'")
genres_df = spark.read.table("movie_silver.genres")

movies_genres_name_df = (
    movies_genres_df.join(genres_df, on="genre_id", how="inner")
    .select("movie_id", "genre_name", movies_genres_df.genre_id)
)
display(movies_genres_name_df)

movie_id,genre_name,genre_id
5,Comedy,35
5,Crime,80
11,Adventure,12
11,Action,28
11,Science Fiction,878
12,Animation,16
12,Family,10751
13,Drama,18
13,Comedy,35
13,Romance,10749


## 5. Añadir `género` al DataFrame

In [0]:
movies_final_df = (
    movies_final_df.join(movies_genres_name_df, on="movie_id", how="inner")
    .select("title", "duration_time", "release_date", "vote_average", "language_name", movies_genres_name_df.genre_name, movies_final_df.movie_id, movies_genres_name_df.genre_id, "language_id")
)

display(movies_final_df)

title,duration_time,release_date,vote_average,language_name,genre_name,movie_id,genre_id,language_id
Minority Report,145,2002-06-20,7.1,svenska,Mystery,180,9648,24575
Minority Report,145,2002-06-20,7.1,svenska,Science Fiction,180,878,24575
Minority Report,145,2002-06-20,7.1,svenska,Thriller,180,53,24575
Minority Report,145,2002-06-20,7.1,svenska,Action,180,28,24575
Så som i himmelen,132,2004-09-03,6.9,svenska,Romance,464,10749,24575
Så som i himmelen,132,2004-09-03,6.9,svenska,Music,464,10402,24575
Så som i himmelen,132,2004-09-03,6.9,svenska,Comedy,464,35,24575
Så som i himmelen,132,2004-09-03,6.9,svenska,Drama,464,18,24575
The Beach,119,2000-02-11,6.3,svenska,Romance,1907,10749,24575
The Beach,119,2000-02-11,6.3,svenska,Thriller,1907,53,24575


## 6. Añadir campo `fecha de creación`

In [0]:
from pyspark.sql.functions import current_timestamp, lit
movies_final_df = ( movies_final_df
                   .withColumn("created_date", lit(v_file_date))
                   .orderBy(movies_final_df.release_date.desc())
)


In [0]:
display(movies_final_df)

title,duration_time,release_date,vote_average,language_name,genre_name,movie_id,genre_id,language_id,created_date
Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Crime,189,80,24574,2024-12-16
Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Thriller,189,53,24574,2024-12-16
Scary Movie 5,86,2013-04-11,4.6,English,Comedy,4258,35,24574,2024-12-16
The Amazing Spider-Man,136,2012-06-27,6.5,English,Action,1930,28,24574,2024-12-16
The Amazing Spider-Man,136,2012-06-27,6.5,English,Fantasy,1930,14,24574,2024-12-16
The Amazing Spider-Man,136,2012-06-27,6.5,English,Adventure,1930,12,24574,2024-12-16
The Adventures of Tintin,107,2011-10-25,6.7,English,Mystery,17578,9648,24574,2024-12-16
The Adventures of Tintin,107,2011-10-25,6.7,English,Animation,17578,16,24574,2024-12-16
The Adventures of Tintin,107,2011-10-25,6.7,English,Adventure,17578,12,24574,2024-12-16
The Ides of March,101,2011-09-24,6.5,English,Drama,10316,18,24574,2024-12-16


## 7. Escribir datos en el DataLake en formato `Delta`

In [0]:
merge_delta_lake( movies_final_df, "movie_gold", "results_movie_genre_language", "tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.genre_id = src.genre_id AND tgt.created_date = src.created_date", "created_date" )

In [0]:
%sql
SELECT * FROM movie_gold.results_movie_genre_language

title,duration_time,release_date,vote_average,language_name,genre_name,movie_id,genre_id,language_id,created_date
Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Crime,189,80,24574,2024-12-16
Sin City: A Dame to Kill For,102,2014-08-20,6.3,English,Thriller,189,53,24574,2024-12-16
Scary Movie 5,86,2013-04-11,4.6,English,Comedy,4258,35,24574,2024-12-16
The Amazing Spider-Man,136,2012-06-27,6.5,English,Action,1930,28,24574,2024-12-16
The Amazing Spider-Man,136,2012-06-27,6.5,English,Fantasy,1930,14,24574,2024-12-16
The Amazing Spider-Man,136,2012-06-27,6.5,English,Adventure,1930,12,24574,2024-12-16
The Adventures of Tintin,107,2011-10-25,6.7,English,Mystery,17578,9648,24574,2024-12-16
The Adventures of Tintin,107,2011-10-25,6.7,English,Animation,17578,16,24574,2024-12-16
The Adventures of Tintin,107,2011-10-25,6.7,English,Adventure,17578,12,24574,2024-12-16
The Ides of March,101,2011-09-24,6.5,English,Drama,10316,18,24574,2024-12-16
